# Appendix. The gene selection comparison study

Question: did any of four ways of choosing a gene list score higher than the others, and did a chosen list add to the clinical covariates?

What enters: the saved tables and settings of the study, computed on the earlier data version (851 development patients) by the archived notebook at commit f8c4a00; nothing is recomputed.

What leaves: nothing new. The supervisors chose elastic-net Cox as the selector afterwards (22 October 2026).

Result: rendered under the imports from the saved tables.

How to run: the nbconvert command of notebook 1 with this file; seconds.

In [1]:
import hashlib
import json
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

HC = "Harrell's C (mean)"
RUNS = Path("../results/appendix_gene_selection_comparison/runs")
TABLES = Path("../results/appendix_gene_selection_comparison/tables")

In [2]:
compared = pd.read_csv(RUNS / "what_is_compared.csv", index_col=0)
methods = list(compared.index[compared["role"] == "selection method"])
genes_only = pd.read_csv(TABLES / "method_comparison_genes_only.csv", index_col=0)
plus_clinical = pd.read_csv(TABLES / "method_comparison_genes_plus_clinical.csv", index_col=0)
against_best = pd.read_csv(TABLES / "method_comparison_against_best.csv", index_col=0)
tied = against_best[(against_best["interval low"] <= 0) & (against_best["interval high"] >= 0)]
best = genes_only.loc[methods, HC].idxmax()
display(Markdown(f"Result: the four methods are tied with genes only (highest mean C {best} at "
                 f"{genes_only.loc[best, HC]:.3f}, {len(tied)} others within its interval); no "
                 f"selected list added to the clinical covariates (clinical only "
                 f"{plus_clinical.loc['clinical only', HC]:.3f})."))

Result: the four methods are tied with genes only (highest mean C elastic-net Cox at 0.659, 3 others within its interval); no selected list added to the clinical covariates (clinical only 0.740).

## What was compared

Two reference rows without genes and four selection methods, each tuned inside the inner folds of a nested cross-validation and scored by a ridge Cox model on the outer test parts, with genes only and with genes plus the clinical covariates.

In [3]:
compared

,what it does,uses patient outcomes to pick genes,role
clinical only,"no gene: a Cox model on age, stage, ER, PR and...",picks no genes,reference
"clinical only, unknown values replaced","as clinical only, with every unknown value rep...",picks no genes,reference
most variable genes,ranks genes by the variance of their expressio...,no,selection method
one-gene Cox ranking,ranks genes by a test of each gene alone in a ...,yes,selection method
LASSO-Cox,fits one Cox model on all genes with a penalty...,yes,selection method
elastic-net Cox,"as LASSO-Cox, with a mixed penalty that lets c...",yes,selection method
all genes,no selection: the scoring model on every gene ...,picks no genes,reference
the earlier pipeline's selector,"LASSO on survival classes, as in the earlier p...","yes, as classes",not run


In [4]:
outcome_driven = compared.loc[methods, "uses patient outcomes to pick genes"] == "yes"
display(Markdown(f"{int(outcome_driven.sum())} of the {len(methods)} selection methods use the "
                 "outcome to pick genes; the references pick no genes."))

3 of the 4 selection methods use the outcome to pick genes; the references pick no genes.

## Settings of the runs

The settings as saved beside the runs, one row per setting; the saved file also says what each one is.

In [5]:
settings = json.loads((RUNS / "settings.json").read_text())
settings_table = pd.DataFrame(settings).T[["value"]]
settings_table

,value
outer folds,5
outer seeds,"[1, 2, 3, 4, 5]"
inner folds,5
inner seed,0
list sizes,"[50, 100, 200, 500, 1000, 2000]"
L1 shares,"[0.25, 0.5, 0.75]"
path length,50
path depth,0.01
longest list,2000
ridge grid,"[1.0, 10.0, 100.0, 1000.0, 10000.0]"


## Results of both scoring forms

Mean Harrell's C over the outer folds, its standard deviation, Uno's C and the median list size, with genes only and with genes plus the clinical covariates.

In [6]:
display(genes_only.round(3), plus_clinical.round(3))

,Harrell's C (mean),SD over the folds,Uno's C (mean),genes in the list (median)
most variable genes,0.657,0.059,0.662,2000.0
one-gene Cox ranking,0.651,0.062,0.664,2000.0
LASSO-Cox,0.649,0.067,0.653,174.0
elastic-net Cox,0.659,0.073,0.663,384.0
all genes,0.689,0.066,0.709,14321.0


,Harrell's C (mean),SD over the folds,Uno's C (mean),genes in the list (median)
clinical only,0.740,0.052,0.699,0.0
"clinical only, unknown values replaced",0.721,0.062,0.675,0.0
most variable genes,0.730,0.061,0.690,100.0
one-gene Cox ranking,0.721,0.069,0.687,50.0
LASSO-Cox,0.733,0.057,0.701,162.0
elastic-net Cox,0.728,0.063,0.697,265.0
all genes,0.716,0.074,0.702,14321.0


In [7]:
display(Markdown(f"With genes only the highest mean among the methods is {best} at "
                 f"{genes_only.loc[best, HC]:.3f}; with the clinical covariates no "
                 f"method rises above clinical only at "
                 f"{plus_clinical.loc['clinical only', HC]:.3f}."))

With genes only the highest mean among the methods is elastic-net Cox at 0.659; with the clinical covariates no method rises above clinical only at 0.740.

## Paired comparisons on both metrics

Each row is a difference between two rows scored on the same outer folds, with its 95% interval from a corrected resampled t over 25 folds; an interval that includes 0 shows no difference.

In [8]:
paired = pd.read_csv(TABLES / "paired_comparisons.csv", index_col=0)
paired

,Harrell's C: difference (low to high),"Uno's C, secondary: difference (low to high)",fixed in advance or exploratory
"elastic-net Cox minus LASSO-Cox, genes only",+0.010 (-0.039 to +0.058),+0.010 (-0.041 to +0.061),fixed in advance
most variable genes plus clinical minus clinical only,-0.009 (-0.049 to +0.030),-0.008 (-0.050 to +0.033),fixed in advance
one-gene Cox ranking plus clinical minus clinical only,-0.019 (-0.063 to +0.024),-0.011 (-0.052 to +0.029),fixed in advance
LASSO-Cox plus clinical minus clinical only,-0.007 (-0.042 to +0.029),+0.002 (-0.036 to +0.040),fixed in advance
elastic-net Cox plus clinical minus clinical only,-0.011 (-0.056 to +0.034),-0.001 (-0.049 to +0.047),fixed in advance
all genes plus clinical minus clinical only,-0.024 (-0.078 to +0.031),+0.004 (-0.056 to +0.064),exploratory
"all genes, genes only, minus clinical only",-0.051 (-0.128 to +0.027),+0.011 (-0.077 to +0.099),exploratory
"all genes minus most variable genes, genes only",+0.032 (-0.023 to +0.087),+0.047 (-0.009 to +0.104),exploratory
"clinical only, unknown values replaced, minus clinical only",-0.019 (-0.049 to +0.012),-0.024 (-0.063 to +0.016),exploratory


In [9]:
both = pd.read_csv(TABLES / "both_metrics.csv", index_col=0)
includes = int(both["Harrell's C: interval includes 0"].sum())
display(Markdown(f"{includes} of the {len(both)} intervals on Harrell's C include 0, and "
                 f"{int(both[both.columns[-2]].sum())} of {len(both)} on Uno's C."))

8 of the 8 intervals on Harrell's C include 0, and 8 of 8 on Uno's C.

## Leakage check

The same methods with the gene list chosen on all patients before the folds, against selection inside the training fold; the difference is the score that leaks in.

In [10]:
leakage = pd.read_csv(TABLES / "leakage_check_table.csv", index_col=0)
leakage.round(3)

,"C, selection on all patients","C, selection inside the training fold",difference,interval low,interval high,"genes, selection on all patients","genes, selection inside the training fold (median)"
one-gene Cox ranking,0.725,0.606,0.119,0.075,0.163,100,100.0
LASSO-Cox,0.904,0.649,0.255,0.184,0.326,97,174.0
elastic-net Cox,0.968,0.659,0.309,0.235,0.383,1377,384.0


In [11]:
display(Markdown(f"Selecting genes on all patients inflates Harrell's C by "
                 f"{leakage['difference'].min():+.3f} to {leakage['difference'].max():+.3f}."))

Selecting genes on all patients inflates Harrell's C by +0.119 to +0.309.

## The choice rule

The rule fixed before the runs: take the highest mean with genes only, count as tied every method whose interval against it includes 0, and among the tied prefer the method that does not use the outcome.

In [12]:
uses_outcome = compared.loc[tied.index, "uses patient outcomes to pick genes"]
no_outcome = list(uses_outcome.index[uses_outcome == "no"])
display(Markdown(f"Highest mean with genes only: {best}. Tied with it: {', '.join(tied.index)}. "
                 f"The rule picks {', '.join(no_outcome)}. The supervisors chose elastic-net Cox "
                 "afterwards, which this study had found tied with the others."))

Highest mean with genes only: elastic-net Cox. Tied with it: most variable genes, one-gene Cox ranking, LASSO-Cox. The rule picks most variable genes. The supervisors chose elastic-net Cox afterwards, which this study had found tied with the others.

## Run record

The commit of this notebook, the package versions, and the commit and hashes recorded by the archived runs notebook.

In [13]:
def git(*arguments):
    """The output of one git command, for the run record."""
    command = ["git", *arguments]
    return subprocess.run(command, capture_output=True, text=True, check=True).stdout.strip()


runs_record = json.loads((RUNS / "run_record.json").read_text())
tables_hash = hashlib.sha256()
for path in sorted(TABLES.glob("*.csv")):
    tables_hash.update(path.read_bytes())
run_record = {"git commit": git("rev-parse", "HEAD"),
              "tracked files changed since that commit":
                  git("status", "--porcelain", "--untracked-files=no").splitlines(),
              "python": sys.version.split()[0], "pandas": version("pandas"),
              "runs computed at commit": runs_record.get("git commit"),
              "saved tables (sha256 of all csv files)": tables_hash.hexdigest()}
print(json.dumps(run_record, indent=1))

{
 "git commit": "3ef0dbda36e788b220bedc750c1838e9aee5bb95",
 "tracked files changed since that commit": [],
 "python": "3.12.14",
 "pandas": "3.0.6",
 "runs computed at commit": "61198544ab61f1dfda933954c2c0a49236019af3",
 "saved tables (sha256 of all csv files)": "343cf77f90532a5fee277e0e7390d6a1cced5831af4ffb1ad62974a1e5aea893"
}
